# 03 — Gold Layer: Analytical Aggregations

This notebook demonstrates the **Gold layer** — the consumption-ready analytical tables.

**Tables produced:**
| Table | Grain | Description |
|---|---|---|
| `daily_metrics` | ticker × trade_date | OHLCV + rolling volatility, avg volume, cumulative return |
| `portfolio_summary` | ticker | Period totals: total return, avg volume, avg volatility |
| `monthly_returns` | ticker × year × month | Monthly OHLC and return |

In [ ]:
# Configuração do notebook da camada Gold (analytics).
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
pl.Config.set_tbl_rows(20)

## 1. Build Gold tables

In [ ]:
# 1) Constrói as 3 tabelas analíticas a partir da Silver:
#    - daily_metrics       (ticker × trade_date, com features de rolling window)
#    - portfolio_summary   (1 linha por ticker)
#    - monthly_returns     (ticker × ano × mês)
from f_pipelines.gold_pipeline import GoldPipeline

tables = GoldPipeline().run()

for name, df in tables.items():
    print(f"{name:25s}: {len(df):>6,} rows  |  cols: {df.columns}")

## 2. Daily metrics overview

In [ ]:
# 2a) Amostra da tabela `daily_metrics`.
# Aqui aparecem as features derivadas: avg_volume_20d, volatility_20d, cum_return.
daily = tables.get("daily_metrics")
if daily is not None:
    daily.head(10)

In [ ]:
# 2b) Comparativo de retorno acumulado entre os 5 ativos demo.
# Indicador clássico para "quanto você teria ganho/perdido se tivesse
# investido R$1 no início do período".
if daily is not None:
    top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
    plot_df = daily.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

    fig = px.line(
        plot_df,
        x="trade_date",
        y="cum_return",
        color="ticker",
        title="Retorno Acumulado por Ativo — Camada Gold",
        labels={
            "trade_date": "Data do Pregão",
            "cum_return": "Retorno Acumulado",
            "ticker": "Ativo",
        },
    )
    fig.add_hline(y=0, line_dash="dash", line_color="grey")
    fig.update_layout(
        height=520,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
        margin=dict(l=60, r=40, t=70, b=50),
    )
    fig.update_yaxes(tickformat=".1%")
    fig.update_xaxes(tickformat="%d/%m/%Y")
    fig.show()

In [ ]:
# 2c) Volatilidade anualizada com janela móvel de 20 pregões.
# Fórmula: std(daily_return, 20) × sqrt(252) — onde 252 é o nº médio de pregões/ano.
if daily is not None:
    fig2 = px.line(
        plot_df,
        x="trade_date",
        y="volatility_20d",
        color="ticker",
        title="Volatilidade Anualizada — Janela Móvel de 20 dias",
        labels={
            "trade_date": "Data do Pregão",
            "volatility_20d": "Volatilidade Anualizada",
            "ticker": "Ativo",
        },
    )
    fig2.update_layout(
        height=520,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
        margin=dict(l=60, r=40, t=70, b=50),
    )
    fig2.update_yaxes(tickformat=".1%")
    fig2.update_xaxes(tickformat="%d/%m/%Y")
    fig2.show()

## 3. Portfolio summary

In [ ]:
# 3a) Ranking dos ativos pelo retorno total no período.
summary = tables.get("portfolio_summary")
if summary is not None:
    print("Top performers:")
    display(summary.head(10).to_pandas())

In [ ]:
# 3b) Dispersão Risco × Retorno.
# Cada ponto é um ativo; eixo X = volatilidade média, eixo Y = retorno total.
# Quadrante superior-esquerdo = melhor relação (alto retorno, baixo risco).
if summary is not None:
    sum_pd = summary.to_pandas()
    fig3 = px.scatter(
        sum_pd.dropna(subset=["total_return", "avg_volatility"]),
        x="avg_volatility",
        y="total_return",
        text="ticker",
        title="Risco x Retorno no Período (por Ativo)",
        labels={
            "avg_volatility": "Volatilidade Média (anualizada)",
            "total_return": "Retorno Total no Período",
            "ticker": "Ativo",
        },
        color="ticker",
    )
    fig3.update_traces(textposition="top center", marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))
    fig3.update_layout(
        height=560,
        template="plotly_white",
        showlegend=False,
        margin=dict(l=60, r=40, t=70, b=60),
    )
    fig3.add_hline(y=0, line_dash="dash", line_color="grey")
    fig3.update_xaxes(tickformat=".1%")
    fig3.update_yaxes(tickformat=".1%")
    fig3.show()

## 4. Monthly returns heatmap

In [ ]:
# 4a) Amostra da tabela `monthly_returns` (agregação mensal).
monthly = tables.get("monthly_returns")
if monthly is not None:
    monthly.head(10)

In [ ]:
# 4b) Mapa de calor: retorno mensal por ativo.
# Eixo X = ano-mês, eixo Y = ativo; cor verde = ganho, vermelho = perda.
# Excelente para identificar visualmente meses bons/ruins de cada ativo.
import seaborn as sns
import matplotlib.pyplot as plt

if monthly is not None:
    # Pivota o long-format para wide (ticker x year-month) para o heatmap.
    pivot = (
        monthly
        .filter(pl.col("ticker").is_in(top5))
        .with_columns(
            pl.concat_str(
                [pl.col("year").cast(pl.Utf8), pl.lit("-"), pl.col("month").cast(pl.Utf8).str.zfill(2)]
            ).alias("ym")
        )
        .pivot(values="month_return", index="ticker", on="ym")
        .to_pandas()
        .set_index("ticker")
        .astype(float)
    )

    fig4, ax = plt.subplots(figsize=(max(10, 1.1 * pivot.shape[1]), 4.5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1%",
        cmap="RdYlGn",       # divergente: vermelho (perda) → verde (ganho)
        center=0,            # zero como ponto neutro do colormap
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Retorno Mensal"},
        ax=ax,
    )
    ax.set_title("Mapa de Calor — Retorno Mensal por Ativo", fontsize=13, pad=12)
    ax.set_xlabel("Ano-Mês")
    ax.set_ylabel("Ativo")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Read persisted Gold tables

In [ ]:
# 5) Leitura das tabelas Gold persistidas em disco.
# Útil para confirmar a integridade do ciclo write→read em produção.
from d_processing.gold.aggregate import read_gold

gold_daily = read_gold("daily_metrics")
print(f"Persisted daily_metrics rows: {len(gold_daily):,}")